# CR-1a G2 本番の引き直し（Claude バッチ API・CPU）

段3 で固定したリクエスト 226,704 件を、凍結した計画の G2 の決まりどおりにバッチ API で送り、結果を集めます。

- **送る前に**：リクエストの一覧を作り直し、段3 のロックのハッシュ（`9186cb93…`）と Drive の一覧ファイルの両方と一致することを確かめます。違えば 1 件も送りません。
- **送り方**：1 回 10,000 件まで、同時に 3 つまで。次のバッチの前に毎回、計画の式で予算（上限 $90）を見張ります（足りなければ、流れているバッチが戻るのを待つ。流れていないのに足りなければ止める）。失敗したリクエストは同じ番号で 2 回まで出し直します。11/7 以降は新しいバッチを出しません（クレジットの期限 11/8）。
- **費用の見込み**：$55〜66（バッチは半額）。
- **時間**：数時間〜半日くらい（バッチの処理待ち）。セル3 がずっと回り続けます（1 分ごとに確かめます）。
- **途中で切れても大丈夫**：バッチは Anthropic 側で処理が続きます。もう一度セル1 → セル2 → セル3 を実行すれば、集めた結果ファイルを確かめてから、続きを集めて送ります。送ったかどうか分からないバッチがあるときは、それが見つかるまで新しいバッチを出しません（30 分見つからなければ止まるので、Claude に教えてね）。
- **ランタイムは 1 つだけ**（千紗さんの約束で守ります）：G2 の記録には、回しているランタイムの番号が入ります。別のランタイム（Colab が切れて作り直したときなど）でセル2 を実行すると、「前のランタイムが止まっているか確かめて」と止まります。Colab の「ランタイム → セッションを管理」で前のランタイムが残っていないことを確かめたら、セル2 の一番上の `OWNER_CONFIRMS_OLD_RUNTIME_STOPPED` に表示された番号を入れて、もう一度セル2 → セル3。コードも送る前に毎回、記録が自分の書いたままかを確かめ、違えば送らずに止まります（ただしこれは見張りで、鍵ではありません）。
- 問題文・応答は表示しません。応答は Drive の非公開ファイル（`*_PRIVATE.jsonl`）にだけ保存します。

**使い方**：ランタイムは **CPU**。シークレットは `ANTHROPIC_API_KEY` と `GITHUB_TOKEN`（`HF_TOKEN` は要りません）。セル1 → セル2 → セル3。結果は Drive の `DECISION_POINTS/cr1a/G2/` に入ります。全部終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, lock present, pinned SDK, key (never printed), notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, datetime as dt
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
STUDY = "CR-1a"
EXPECTED_HEAD = "f6c6bb94fe2ff77a5726e71b835f7ee0632a9d4d"
G1_HEAD = "7d95ee90a7b536fb85326e2a30981b9bc8030421"
LOCK_COMMIT = "69b43d8"
LOCK_SHA = "9186cb93b31f803ef144497830c3998ca32c04cf0afccca9d868c1ba7bae088d"
G1_ITEMS_SHA = "bd42639245cb234f86e0bcd97092db67ae2842c9f1e48e123a9ee1b484c931b5"
LOCK_BLOB = "549fee56602ffa09ee176911bae5b811811368a7"
ERRATUM_BLOB = "7ec14e4bcf13eae1bee21225f879aebcc9088d55"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
for anc in (G1_HEAD, LOCK_COMMIT, "69dd041"):
    assert subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", anc, head]).returncode == 0, anc + " が祖先ではありません"
token = None
env.pop("GITHUB_TOKEN", None)
def blob(path):
    return subprocess.check_output(["git", "-C", str(REPO), "hash-object", path], text=True).strip()
assert blob("notes/cr1a_lock_pre_g2.md") == LOCK_BLOB and LOCK_SHA in (REPO / "notes/cr1a_lock_pre_g2.md").read_text(), "段3 のロックが違います。G2 は出しません。"
assert blob("notes/cr1a_lock_pre_g2_erratum.md") == ERRATUM_BLOB, "ロックの訂正記録（オーナー確認済み）が違います。G2 は出しません。"
sys.path.insert(0, str(REPO))
from pork import cr1_requests as RQ
from pork import cr1_run as R
from pork import cr1_analysis as A
from pork import cr1_g2 as G
R.check_study(STUDY)
assert RQ.BUDGET_USD == 90.0, "予算の上限が［決定4］の $90 と違います"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic==" + RQ.SDK_VERSION], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
import anthropic
assert anthropic.__version__ == RQ.SDK_VERSION, anthropic.__version__
_key = get_secret("ANTHROPIC_API_KEY")
assert _key, "Colab のシークレットに ANTHROPIC_API_KEY を登録してね。"
client = anthropic.Anthropic(api_key=_key)                       # reads (status / results / list): default retries
create_client = anthropic.Anthropic(api_key=_key, max_retries=0)  # create: never retried by the SDK
del _key
API = G.AnthropicBatchAPI(client, create_client)
TRANSIENT = (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError, anthropic.APITimeoutError)
print("study:", STUDY, "| exact head:", head, "| anthropic", anthropic.__version__)

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-1a G2 本番の引き直し（バッチAPI・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-1a G2 batches', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — rebuild every request and check it against the 段3 lock; open (or resume) the G2 state.
# 前のランタイムが止まっていることを千紗さんが確かめたときだけ、ここにその番号を入れる（ふだんは空のまま）
OWNER_CONFIRMS_OLD_RUNTIME_STOPPED = ""
from pork import cr1_items as CI
rows, DATA_ID = R.load_rows_public()
chain = CI.chain_and_candidates(rows)
del rows
assert CI.candidates_sha256(chain) == R.CANDIDATES_SHA256, "候補の一覧が段2 と違います。止めます。"
D = Path("/content/drive/MyDrive/DECISION_POINTS/cr1a")
ck = json.loads((D / "G1/CR1A_G1_checkpoint_PRIVATE.json").read_text())
assert ck["header"] == R.header(STUDY, G1_HEAD, R.CANDIDATES_SHA256, A.MODEL, RQ.SDK_VERSION)
R.validate(ck, chain)
g1 = R.summarize_g1(ck)
assert g1["verdict"] == "GO" and g1["final_items_ids_sha256"] == G1_ITEMS_SHA
IN_BOUND = G.input_bound_tokens(ck)
assert IN_BOUND <= G.WORST_IN - 20, f"入力の上限の見積もり（{IN_BOUND} トークン）が予算の式の最悪値 600 に近すぎます。止めます。"
print("1 件の入力の目安（G1 の土台の道から）:", IN_BOUND, "トークン（予算の式の最悪値は 600）")
OUT = D / "G2"
drive_manifest = json.loads((OUT / "CR1A_G2_manifest.json").read_text())
REQ = G.Requests(g1["final_items"], ck["items"], chain, STUDY, LOCK_SHA, drive_manifest=drive_manifest)
del ck, drive_manifest
print("リクエスト:", len(REQ), "| 一覧のハッシュはロックと一致")
MAXIN = G.max_input_tokens(REQ, API.count_tokens)                 # token counter (free): longest prefill of each item
assert MAXIN["max"] <= G.WORST_IN, f"入力が {MAXIN['max']} トークンの問題があり、予算の式の最悪値 600 を超えます。止めます。"
print("いちばん長い入力（API のトークン数え）:", MAXIN["max"], "トークン ≤ 600")
R.drive_smoke(OUT)
STATE = OUT / "CR1A_G2_state.json"
HDR = G.g2_header(STUDY, LOCK_SHA, head, anthropic.__version__)
if STATE.exists():
    saved = json.loads(STATE.read_text())
    if saved["header"]["code_head"] != head:      # the same run may be resumed from a later notebook commit only if
        raise G.G2Stop("G2 state was written by another code commit; ask Claude before resuming")
    ST = G.open_state(saved, HDR)
else:
    ST = G.open_state(None, HDR)

def save():
    G.save_state(STATE, ST)                                        # bumps the generation counter

def fence():
    G.fence(ST, lambda: json.loads(STATE.read_text()))             # the state on Drive is still the one we wrote

import uuid
_rid = Path("/content/g2_runtime_id")                              # same VM (kernel restart) keeps its id
if not _rid.exists():
    _rid.write_text(uuid.uuid4().hex[:12])
RUNTIME = _rid.read_text().strip()
JST = dt.timezone(dt.timedelta(hours=9))
print("このランタイムの番号:", RUNTIME)
G.claim_lease(ST, RUNTIME, dt.datetime.now(JST), save, owner_confirms_stopped=OWNER_CONFIRMS_OLD_RUNTIME_STOPPED or None)
n_rec = G.verify_collected(ST, REQ, OUT, api=API, save=save)      # every collected results file: exists, sha, ids, counts
print("集めた結果ファイルの確かめ: OK" + (f"（{n_rec} 個をダウンロードし直して復旧）" if n_rec else ""))
G.resolve_submitting(ST, API, save, dt.datetime.now(JST))
save()
t = G.tallies(ST, REQ)
print("送ったバッチ:", len(ST["batches"]), "| 成功:", len(t["succeeded"]), "| 流れている:", len(t["inflight"]),
      "| 残りのかたまり:", len(t["chunks_left"]), "| 出し直し待ち:", len(t["retry"]))

In [ ]:
# Cell 3 — the G2 loop: collect ended batches, send while the plan allows, every 60 s, until done or stopped.
last = None
while True:
    try:
        fence()
        G.claim_lease(ST, RUNTIME, dt.datetime.now(JST), save)
        G.collect(ST, REQ, API, OUT, save)
        if ST["submitting"]:                       # an uncertain create: look it up; nothing is sent while it is open
            G.resolve_submitting(ST, API, save, dt.datetime.now(JST))
        G.submit_ready(ST, REQ, API, save, dt.datetime.now(JST), fence_check=fence)
    except TRANSIENT as e:                         # G2Stop and any other error stop the cell
        print("一時的なエラー:", type(e).__name__, "→ 1 分後にもう一度（送ったかどうか分からないバッチがあれば、見つかるまで新しいバッチは出しません）")
        time.sleep(60)
        continue
    save()
    if ST["stopped"] or G.done(ST, REQ):
        break
    t = G.tallies(ST, REQ)
    line = (f"成功 {len(t['succeeded'])}/{len(REQ)} | 流れている {len(t['inflight'])} | 残りのかたまり {len(t['chunks_left'])}"
            f" | 出し直し待ち {len(t['retry'])} | 費用 ${G.spent_usd(ST):.2f}")
    if line != last:
        print(dt.datetime.now(JST).strftime("%H:%M"), line)
        last = line
    time.sleep(60)
G.verify_collected(ST, REQ, OUT, api=API, save=save)
S = G.summary(ST, REQ)
S["dataset"] = DATA_ID
S["input_bound_tokens"] = IN_BOUND
R._atomic_write(OUT / "CR1A_G2_summary.json", S)
print("終わり:", "完了" if S["done"] else S["stopped"], "| 成功", S["succeeded"], "| 解決しない", S["unresolved"],
      "| 費用 $%.2f" % S["spent_usd"])
msg = '終わりました：G2 の全バッチを集めました' if S["done"] else '止まりました：計画の決まりで G2 を止めました（Claude に見せてね）'
notify(msg, tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()